In [4]:
# ============================================================
# CELL PURPOSE: Import libraries and define data path
# ============================================================

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

DATA_PATH = "/Users/sanjana/Desktop/Hype-Predictor/Data"

In [6]:
# ============================================================
# CELL PURPOSE: Load all datasets and inspect structure
# ============================================================

# Load CSVs
trends = pd.read_csv(f"{DATA_PATH}/google_trends.csv")
reddit = pd.read_csv(f"{DATA_PATH}/reddit_data.csv")
youtube = pd.read_csv(f"{DATA_PATH}/youtube_data.csv")
news = pd.read_csv(f"{DATA_PATH}/news_data.csv")

# Print column names (CRITICAL RULE)
print("Trends Columns:", trends.columns)
print("Reddit Columns:", reddit.columns)
print("YouTube Columns:", youtube.columns)
print("News Columns:", news.columns)

# Preview data
print(trends.head())
print(reddit.head())
print(youtube.head())
print(news.head())

# ADD THIS (sanity check)
print("\nShapes:")
print("Trends:", trends.shape)
print("Reddit:", reddit.shape)
print("YouTube:", youtube.shape)
print("News:", news.shape)

Trends Columns: Index(['date', 'PS5 Pro', 'iPhone 17', 'Air Jordan 11', 'Owala FreeSip',
       'Nvidia RTX 5090'],
      dtype='object')
Reddit Columns: Index(['product', 'subreddit', 'title', 'score', 'comments', 'upvote_ratio'], dtype='object')
YouTube Columns: Index(['product', 'title', 'view_count', 'like_count', 'comment_count',
       'published_at'],
      dtype='object')
News Columns: Index(['product', 'article_count', 'total_available', 'unique_sources',
       'source_names'],
      dtype='object')
         date  PS5 Pro  iPhone 17  Air Jordan 11  Owala FreeSip  \
0  2021-04-11        2          0             30              0   
1  2021-04-18        2          0             47              0   
2  2021-04-25        2          0             39              0   
3  2021-05-02        2          0             61              0   
4  2021-05-09        2          0             40              0   

   Nvidia RTX 5090  
0                0  
1                0  
2                0 

In [8]:
# ============================================================
# CELL PURPOSE: Reshape Google Trends data
# ============================================================

trends_long = trends.melt(
    id_vars=["date"],
    var_name="product",
    value_name="interest"
)

print(trends_long.head())

         date  product  interest
0  2021-04-11  PS5 Pro         2
1  2021-04-18  PS5 Pro         2
2  2021-04-25  PS5 Pro         2
3  2021-05-02  PS5 Pro         2
4  2021-05-09  PS5 Pro         2


In [10]:
# ============================================================
# Aggregate Google Trends
# ============================================================

trends_agg = trends_long.groupby("product").agg({
    "interest": "mean"
}).reset_index()

trends_agg.rename(columns={"interest": "trends_score"}, inplace=True)

print(trends_agg)

           product  trends_score
0    Air Jordan 11     33.114504
1  Nvidia RTX 5090      9.385496
2    Owala FreeSip     11.656489
3          PS5 Pro     10.225191
4        iPhone 17      4.362595


In [12]:
# ============================================================
# Aggregate Reddit
# ============================================================

reddit_agg = reddit.groupby("product").agg({
    "score": "sum",
    "comments": "sum"
}).reset_index()

reddit_agg["reddit_score"] = reddit_agg["score"] + reddit_agg["comments"]

print(reddit_agg)

           product  score  comments  reddit_score
0    Air Jordan 11   3284      1111          4395
1  Nvidia RTX 5090  14788      6533         21321
2    Owala FreeSip  23348      3323         26671
3          PS5 Pro  32748     34042         66790
4        iPhone 17  25906      6491         32397


In [14]:
# ============================================================
# Aggregate YouTube
# ============================================================

youtube_agg = youtube.groupby("product").agg({
    "view_count": "sum",
    "like_count": "sum",
    "comment_count": "sum"
}).reset_index()

youtube_agg["youtube_score"] = (
    youtube_agg["view_count"] +
    youtube_agg["like_count"] +
    youtube_agg["comment_count"]
)

print(youtube_agg)

           product  view_count  like_count  comment_count  youtube_score
0    Air Jordan 11     2879391      111771           3101        2994263
1  Nvidia RTX 5090    14985368      526589          30091       15542048
2    Owala FreeSip     1839405       22316           1408        1863129
3          PS5 Pro     3563703       64684          14669        3643056
4        iPhone 17     8444558      221355           8938        8674851


In [16]:
# ============================================================
# Aggregate News
# ============================================================

news_agg = news[["product", "article_count"]].copy()
news_agg.rename(columns={"article_count": "news_score"}, inplace=True)

print(news_agg)

           product  news_score
0          PS5 Pro          98
1        iPhone 17          99
2    Air Jordan 11          96
3    Owala FreeSip          27
4  Nvidia RTX 5090         100


In [18]:
# ============================================================
# CELL PURPOSE: Merge all sources
# ============================================================

df = trends_agg.merge(reddit_agg, on="product", how="outer")
df = df.merge(youtube_agg, on="product", how="outer")
df = df.merge(news_agg, on="product", how="outer")

# Fill missing values (raw stage only)
df.fillna(0, inplace=True)

print(df)

           product  trends_score  score  comments  reddit_score  view_count  \
0    Air Jordan 11     33.114504   3284      1111          4395     2879391   
1  Nvidia RTX 5090      9.385496  14788      6533         21321    14985368   
2    Owala FreeSip     11.656489  23348      3323         26671     1839405   
3          PS5 Pro     10.225191  32748     34042         66790     3563703   
4        iPhone 17      4.362595  25906      6491         32397     8444558   

   like_count  comment_count  youtube_score  news_score  
0      111771           3101        2994263          96  
1      526589          30091       15542048         100  
2       22316           1408        1863129          27  
3       64684          14669        3643056          98  
4      221355           8938        8674851          99  


In [ ]:
# ============================================================
# CELL PURPOSE: Save raw merged dataset
# ============================================================

df.to_csv(f"{DATA_PATH}/raw_merged.csv", index=False)

print("✅ raw_merged.csv saved successfully!")